In [53]:
import openai
import pandas as pd
import tiktoken


from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import FieldCondition, MatchAny, Filter
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

### Create Qdrant collection for hybrid search (lexical and contextual)

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance= Distance.COSINE)
    }
)

True

In [5]:
qdrant_client.create_payload_index(
    collection_name = "Amazon-reviews-collection-01",
    field_name = "parent_asin",
    field_schema = PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [6]:
def get_embedding(text, model = "text-embedding-3-small"):
    response = openai.embeddings.create(
        input =text, 
        model=model
    )
    return response.data[0].embedding  

In [7]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Read the sampled dataset with amazon inventory data

In [8]:
df_reviews= pd.read_json("../../data/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [9]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,Nixplay 10.1 touch screen digital picture frame,I purchased this digital frame on a treasure t...,[],B096DQF21Z,B0BNXXNBB4,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-29 06:52:30.702,19,True
1,5,Great so far...,"Speedy delivery, great sound and a great warra...",[],B08H1WNYTR,B0C72D4J46,AFANVB6MPHJTCTFOVIEBKLWZ2GVA,2022-07-05 14:40:48.001,0,True
2,5,Set up is not that easy.,Nice looking set but installation instructions...,[],B09XGX1GMJ,B09XGQQ98K,AHACLF2COQQE2V33ZFXQ7THZOJ2Q,2022-09-21 11:26:42.074,0,True
3,2,Waste of money,"Very unhappy with this keyboard, it would slip...",[],B07899MFZ2,B07L5L22ZL,AGYEAZK4OEYF2MSSTGJ5WNJDVZKA,2018-09-29 22:39:47.708,1,True
4,5,Nice,Work great,[],B09JSMNZRG,B09LTX3SQX,AH67BI7JTOFR35HMZYFVOEHM4CPQ,2023-01-11 21:15:08.805,0,True


In [10]:
len(df_reviews)

122840

### Preprocess title and features

In [16]:
def preprocessed_reviews_data(row):
    return f"{row['title']} {row['text']}"

In [18]:
df_reviews["preprocessed_data"] = df_reviews.apply(preprocessed_reviews_data, axis=1)

In [ ]:
df_reviews.head()

In [ ]:
#tokenizer that the model uses to split text into tokens
def count_tokens(row):
    encoding = tiktoken.encoding_for_model("text-embedding-3-small")

    return len(encoding.encode(row["preprocessed_data"]))

In [22]:
df_reviews["token_count"] = df_reviews.apply(count_tokens, axis=1)

In [23]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,token_count
0,4,Nixplay 10.1 touch screen digital picture frame,I purchased this digital frame on a treasure t...,[],B096DQF21Z,B0BNXXNBB4,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-29 06:52:30.702,19,True,Nixplay 10.1 touch screen digital picture fram...,255
1,5,Great so far...,"Speedy delivery, great sound and a great warra...",[],B08H1WNYTR,B0C72D4J46,AFANVB6MPHJTCTFOVIEBKLWZ2GVA,2022-07-05 14:40:48.001,0,True,"Great so far... Speedy delivery, great sound a...",27
2,5,Set up is not that easy.,Nice looking set but installation instructions...,[],B09XGX1GMJ,B09XGQQ98K,AHACLF2COQQE2V33ZFXQ7THZOJ2Q,2022-09-21 11:26:42.074,0,True,Set up is not that easy. Nice looking set but ...,18
3,2,Waste of money,"Very unhappy with this keyboard, it would slip...",[],B07899MFZ2,B07L5L22ZL,AGYEAZK4OEYF2MSSTGJ5WNJDVZKA,2018-09-29 22:39:47.708,1,True,Waste of money Very unhappy with this keyboard...,66
4,5,Nice,Work great,[],B09JSMNZRG,B09LTX3SQX,AH67BI7JTOFR35HMZYFVOEHM4CPQ,2023-01-11 21:15:08.805,0,True,Nice Work great,3


In [24]:
len(df_reviews)

122840

In [ ]:
#filter out anything longer than expected amount of token
df_reviews = df_reviews[df_reviews["token_count"] < 8192]

In [25]:
len(df_reviews)

122840

In [26]:
#total tokens to be embed
total_tokens = df_reviews["token_count"].sum()


In [27]:
total_tokens

np.int64(6506026)

### Embed the text and add additional fields to the payload of each vector for reviews

In [28]:
df_data_to_embed = df_reviews[["preprocessed_data", "parent_asin"]]

In [29]:
df_data_to_embed.head()

,preprocessed_data,parent_asin
0,Nixplay 10.1 touch screen digital picture fram...,B0BNXXNBB4
1,"Great so far... Speedy delivery, great sound a...",B0C72D4J46
2,Set up is not that easy. Nice looking set but ...,B09XGQQ98K
3,Waste of money Very unhappy with this keyboard...,B07L5L22ZL
4,Nice Work great,B09LTX3SQX


In [30]:
data_to_embed_reviews = df_data_to_embed.to_dict(orient="records")

In [ ]:
data_to_embed_reviews

In [33]:
len(data_to_embed_reviews)

122840

In [41]:
text_to_embed_reviews = [item["preprocessed_data"] for item in data_to_embed_reviews]

In [ ]:
text_to_embed_reviews

In [ ]:
embeddings = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

In [47]:
len(embeddings)

122840

In [48]:
pointstructs = []
i=1
for embedding, data in zip(embeddings, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
            },
            payload=data
        )
    )
    i += 1

### A function to run search against reviews on a prefiltered set of product IDs

In [54]:
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )
    
    return results

In [56]:
reviews = retrieve_prefiltered_reviews_data("bad_quality", ["B09Q5TNDHY"])

In [58]:
reviews.points

[ScoredPoint(id=62005, version=625, score=0.5, payload={'preprocessed_data': 'Its a total waste and the screen came as if it was used before The screen unlike the picture is so small and when i took it out of the box it looks like the watch was used before its not new<br />wouldn’t recommend', 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=80491, version=809, score=0.33333334, payload={'preprocessed_data': "Its stopped working It's not that durable", 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=43190, version=434, score=0.25, payload={'preprocessed_data': "Doesn't work step tracker to sensitive It's nice, decent features good screen... but step accuracy is way off . Did the dishes and said I walked 100 steps. Also walked 100 steps in my sleep", 'parent_asin': 'B09Q5TNDHY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=105619, version=1061, score=0.2, payload={'preprocessed

In [59]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B09Q5TNDHY", "B0B4NJ8NKN"])

In [ ]:
reviews.points

### Define the reviews retrieval tool

In [60]:
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    qdrant_client = QdrantClient(url="http://localhost:6333")

    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_data"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
    }


def process_reviews_context(context):

    formatted_context = ""

    for id, chunk in zip(context["retrieved_context_ids"], context["retrieved_context"]):
        formatted_context += f"- ID: {id}, user review: {chunk}\n"

    return formatted_context


def get_formatted_reviews_context(query: str, parent_asins: list[str], top_k: int = 5) -> str:

    """Get the top k reviews matching a query for a list of prefiltered items.
    
    Args:
        query: The query to get the top k reviews for
        item_list: The list of item IDs to prefilter for before running the query
        top_k: The number of reviews to retrieve, this should be at least 20 if multipple items are prefiltered
    
    Returns:
        A string of the top k context chunks with IDs prepending each chunk, each representing a review for a given inventory item for a given query.
    """

    retrieved_context = retrieve_prefiltered_reviews_data(
        query,
        parent_asins,
        top_k
    )
    formatted_context = process_reviews_context(retrieved_context)

    return formatted_context

In [ ]:
result = get_formatted_reviews_context("bad quality", ["B09Q5TNDHY", "B0B4NJ8NKN"])

In [62]:
print(result)

- ID: B0B4NJ8NKN, user review: Low quality piece of trash Adapters broke within 2 weeks of usage. I wouldn’t recommend these.
- ID: B0B4NJ8NKN, user review: Buena Buena
- ID: B09Q5TNDHY, user review: Its a total waste and the screen came as if it was used before The screen unlike the picture is so small and when i took it out of the box it looks like the watch was used before its not new<br />wouldn’t recommend
- ID: B0B4NJ8NKN, user review: Defective product. The charger side does not charge. Unfortunately, I did not use them within the return window time period. Don’t waste your money
- ID: B0B4NJ8NKN, user review: Quality product Very nice quality for price, would buy again

